# 03 · Model Evaluation and Analysis

Evaluate the humanitarian `all_data_en` classification models
(TF-IDF + Logistic Regression and BERT).

This notebook:

- Evaluates model performance using accuracy, precision, recall, and F1-score
- Compares performance across different crisis events
- Generates confusion matrices and classification reports
- Identifies and analyzes incorrectly classified posts
- Saves evaluation results and reports in `results/`

In [2]:
import json
import re
import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline

try:
    from IPython.display import display
except ImportError:  # lets the notebook code also run as a plain script
    display = print

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("Evaluation libraries imported successfully")

Evaluation libraries imported successfully


### Set Up Folder Paths

Find the project root and define where data, models, and results live.

- If the notebook runs from `notebooks/`, the project root is one level up.
- Tables go to `results/tables/` and plots to `results/figures/`.
- The baseline model and its settings come from `05_baseline_model.ipynb`.
- BERT is searched for in two folders (saved by `04_bert_experiments.ipynb`). The first one with model weights is used.

In [3]:
from pathlib import Path

ROOT = Path.cwd().resolve()

if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent

DATA = ROOT / "data/processed"
RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"
FIGURES = RESULTS / "figures"

for folder in [RESULTS, TABLES, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)

BASELINE_PATH = ROOT / "checkpoints/baseline/selected_pipeline.joblib"
BASELINE_METADATA = TABLES / "baseline_metadata.json"

BERT_DIRS = [
    ROOT / "models/bert_crisis_classifier",
    ROOT / "notebooks/models/bert_crisis_classifier",
]

print("Project root:", ROOT)
print("Processed data:", DATA)
print("Results folder:", RESULTS)

Project root: /workspaces/cs1684-crisis-nlp
Processed data: /workspaces/cs1684-crisis-nlp/data/processed
Results folder: /workspaces/cs1684-crisis-nlp/results


## 0. Settings

- `EVAL_SPLIT`: which split to score. Keep it on `"dev"` until the final run, then use `"test"` once.
- `MIN_EVENT_POSTS`: events with fewer posts are left out of the per-event tables.
- `RUN_CROSS_CRISIS`: retrains the baseline once per held-out disaster type (takes several minutes).
- `SEED`: random seed for sampling and bootstrapping.

In [4]:
EVAL_SPLIT = "dev"
MIN_EVENT_POSTS = 50
RUN_CROSS_CRISIS = True
SEED = 42

assert EVAL_SPLIT in {"dev", "test"}
print("Evaluating on:", EVAL_SPLIT)

Evaluating on: dev


## 1. Load Processed Data

Load the preprocessed humanitarian datasets to examine their structure and prepare for model evaluation.

In [5]:
def load_split(name):
    df = pd.read_csv(DATA / f"humanitarian_{name}.tsv", sep="\t")
    df["text_clean"] = df["text_clean"].fillna("").astype(str)
    df["id"] = df["id"].astype(str)
    return df.reset_index(drop=True)

train_df = pd.read_csv(DATA / "humanitarian_train.tsv", sep="\t")
dev_df = pd.read_csv(DATA / "humanitarian_dev.tsv", sep="\t")
test_df = pd.read_csv(DATA / "humanitarian_test.tsv", sep="\t")
eval_df = {"dev": dev_df, "test": test_df}[EVAL_SPLIT].copy()

LABELS = sorted(train_df["class_label"].unique())

print("Training samples:", len(train_df))
print("Validation samples:", len(dev_df))
print("Testing samples:", len(test_df))
print(f"Evaluation split ({EVAL_SPLIT}):", len(eval_df))
print("Number of labels:", len(LABELS))

print("\nDataset columns:")
print(test_df.columns.tolist())

print("\nFirst 5 testing samples:")
display(test_df.head())

Training samples: 61164
Validation samples: 8934
Testing samples: 17356
Evaluation split (dev): 8934
Number of labels: 11

Dataset columns:
['id', 'event', 'source', 'text', 'lang', 'lang_conf', 'class_label', 'split', 'source_row', 'text_clean', 'event_group', 'possible_encoding_issue']

First 5 testing samples:


,id,event,source,text,lang,lang_conf,class_label,split,source_row,text_clean,event_group,possible_encoding_issue
0,21767,disaster_events,drd-figureeight-multimedia,Staff at our feeding centre say chronic malnourishment can cause long-term developmental damage and is often difficu...,en,1.0,requests_or_needs,test,1,Staff at our feeding centre say chronic malnourishment can cause long-term developmental damage and is often difficu...,disaster_events,False
1,324921639701716993,2013_west_texas,crisislext6,@johnnyjeff05 you comin down for the summer semesters right,en,1.0,not_humanitarian,test,2,USER you comin down for the summer semesters right,2013_west_texas,False
2,262348997677686785,2012_sandy_hurricane-ontopic,crisislext6,@Nooram_M yea it's upstate I'm like a few hours away.,en,1.0,not_humanitarian,test,3,USER yea it's upstate I'm like a few hours away.,2012_sandy_hurricane,False
3,508341453676756992,2014_pakistan_floods,crisisnlp-cf,"RT @GDLanglands: Teach every Pakistani that it is not enough to solve our own personal problems, country's problems ...",en,1.0,not_humanitarian,test,4,"RT USER : Teach every Pakistani that it is not enough to solve our own personal problems, country's problems are our...",2014_pakistan_floods,True
4,540037166374866944,2014_philippines_typhoon,crisisnlp-cf,"RT @TedWinnerCNN: Stay with @CNN for live cvg as Typhoon Hagupit slams into Philippines, huge storm surge expected, ...",en,1.0,displaced_and_evacuations,test,5,"RT USER : Stay with USER for live cvg as Typhoon Hagupit slams into Philippines, huge storm surge expected, 600k + e...",2014_philippines_typhoon,True


## 2. Get Predictions From Each Model

For each model we first look for saved predictions in `results/tables/<model>_<split>_predictions.tsv`.
If there are none, we load the model, predict, and save the predictions.
Saved predictions are only reused if they line up row by row with the evaluation data.

If the BERT config only has generic label names (`LABEL_0`, `LABEL_1`, ...), the notebook uses the
sorted label order from `04_bert_experiments.ipynb`.

In [6]:
PRED_COLUMNS = ["id", "event", "event_group", "text_clean", "class_label"]


def predictions_path(model_name, split):
    return TABLES / f"{model_name}_{split}_predictions.tsv"


def load_saved_predictions(model_name, split, df):
    path = predictions_path(model_name, split)
    if not path.exists():
        return None
    saved = pd.read_csv(path, sep="\t", dtype={"id": str})
    if "predicted_label" not in saved.columns or len(saved) != len(df):
        warnings.warn(f"{path.name} does not match the {split} data. Predicting again.")
        return None
    if not (saved["id"].astype(str).to_numpy() == df["id"].to_numpy()).all():
        warnings.warn(f"{path.name} rows are in a different order. Predicting again.")
        return None
    return saved["predicted_label"].astype(str).to_numpy()


def save_predictions(model_name, split, df, predictions):
    out = df[[c for c in PRED_COLUMNS if c in df.columns]].copy()
    out["predicted_label"] = predictions
    out["correct"] = out["class_label"].eq(out["predicted_label"])
    out.to_csv(predictions_path(model_name, split), sep="\t", index=False)

In [7]:
def baseline_predictions(df, split):
    saved = load_saved_predictions("baseline", split, df)
    if saved is not None:
        return saved, "loaded saved predictions"
    if not BASELINE_PATH.exists():
        return None, f"no saved predictions and no model at {BASELINE_PATH.relative_to(ROOT)}"
    model = joblib.load(BASELINE_PATH)
    predictions = model.predict(df["text_clean"])
    save_predictions("baseline", split, df, predictions)
    return predictions, "predicted with saved pipeline"

In [8]:
def find_bert_dir():
    weight_files = ["model.safetensors", "pytorch_model.bin"]
    for folder in BERT_DIRS:
        has_config = (folder / "config.json").exists()
        has_weights = any((folder / f).exists() for f in weight_files)
        if has_config and has_weights:
            return folder
    return None


def bert_predict(texts, model_dir, batch_size=64, max_length=128):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    model.to(device).eval()

    id2label = {int(k): v for k, v in model.config.id2label.items()}
    if set(id2label.values()) != set(LABELS):
        warnings.warn("BERT config has no label names. Using sorted label order.")
        id2label = {i: label for i, label in enumerate(LABELS)}

    texts = list(texts)
    predicted_ids = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    with torch.no_grad():
        for b, start in enumerate(range(0, len(texts), batch_size), start=1):
            batch = tokenizer(
                texts[start:start + batch_size],
                truncation=True,
                padding=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            logits = model(**batch).logits
            predicted_ids.extend(logits.argmax(dim=-1).cpu().tolist())
            if b % 50 == 0 or b == n_batches:
                print(f"BERT: {b}/{n_batches} batches")
    return np.array([id2label[i] for i in predicted_ids])


def bert_predictions(df, split):
    saved = load_saved_predictions("bert", split, df)
    if saved is not None:
        return saved, "loaded saved predictions"
    model_dir = find_bert_dir()
    if model_dir is None:
        return None, "no saved predictions and no BERT weights found"
    try:
        predictions = bert_predict(df["text_clean"], model_dir)
    except ImportError as error:
        return None, f"torch/transformers not installed ({error})"
    save_predictions("bert", split, df, predictions)
    return predictions, f"predicted with {model_dir.relative_to(ROOT)}"

In [10]:
MODEL_NAMES = {"baseline": "TF-IDF + LR", "bert": "BERT"}

predictions = {}
for key, get_predictions in [("baseline", baseline_predictions), ("bert", bert_predictions)]:
    preds, status = get_predictions(eval_df, EVAL_SPLIT)
    print(f"{MODEL_NAMES[key]:<12} {status}")
    if preds is not None:
        predictions[key] = preds

if not predictions:
    raise RuntimeError("No model predictions available. Run 05_baseline_model or 04_bert_experiments first.")

for key, preds in predictions.items():
    eval_df[f"pred_{key}"] = preds
    eval_df[f"correct_{key}"] = eval_df["class_label"].eq(eval_df[f"pred_{key}"])

MODELS = list(predictions)
print("\nModels in this evaluation:", [MODEL_NAMES[m] for m in MODELS])

NameError: name 'warnings' is not defined